# BioJEPA v0.6 Data Prep - Notebook 3: Shard Generation

This notebook handles:
1. Processing each dataset incrementally (chunked for large ones)
2. Normalizing expression (CPM + log1p)
3. Building control banks per batch/gem_group
4. Writing pretraining shards
5. Writing training shards with multi-pert format
6. Writing alignment pairs

**Inputs (from Notebooks 1 & 2):**
- `gene_to_idx.json`, `gene_names.json` - gene universe
- `dataset_gene_masks.json` - per-dataset gene masks
- `dataset_splits.json` - train/val/test perturbation sets
- `holdout_perturbations.json` - test split genes to exclude
- `pert_embd/seq_banks/*.npy` - DNA/chemical embeddings
- `pert_embd/target_banks/*.npy` - protein target embeddings

**Outputs:**
- `pretraining/{train,val}/pt_*.npz`
- `training/{train,val,test}/shard_*.npz`
- `pert_embd/{train,val,test}/align_*.npz`

In [ ]:
from pathlib import Path
from scipy.sparse import issparse
from tqdm import tqdm
import pandas as pd
import numpy as np
import scanpy as sc
import json
import gc

In [ ]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')

pretraining_dir = data_dir / 'pretraining'
training_dir = data_dir / 'training'
pert_embd_dir = data_dir / 'pert_embd'

for split in ['train', 'val', 'test']:
    (pretraining_dir / split).mkdir(parents=True, exist_ok=True)
    (training_dir / split).mkdir(parents=True, exist_ok=True)
    (pert_embd_dir / split).mkdir(parents=True, exist_ok=True)

In [ ]:
CHUNK_SIZE = 50000
SHARD_SIZE = 10000
PT_SHARD_SIZE = 50000
COUNT_NORMALIZE_TARGET = 1e4
MAX_N_PERT = 4
SEED = 42
np.random.seed(SEED)

## Load Metadata from Previous Notebooks

In [ ]:
with open(data_dir / 'gene_to_idx.json') as f:
    gene_to_idx = json.load(f)

with open(data_dir / 'gene_names.json') as f:
    gene_names = json.load(f)

with open(data_dir / 'dataset_gene_masks.json') as f:
    dataset_gene_masks = {k: np.array(v, dtype=bool) for k, v in json.load(f).items()}

with open(data_dir / 'dataset_splits.json') as f:
    dataset_splits = json.load(f)

with open(data_dir / 'holdout_perturbations.json') as f:
    holdout_perts = set(json.load(f))

with open(data_dir / 'cell_type_to_id.json') as f:
    cell_type_to_id = json.load(f)

symbol_to_ensg = {}
for ensg, idx in gene_to_idx.items():
    if idx < len(gene_names):
        symbol_to_ensg[gene_names[idx]] = ensg

N_GENES = len(gene_to_idx)
print(f'Gene universe: {N_GENES}')
print(f'Holdout perturbations: {len(holdout_perts)}')
print(f'Gene symbol -> ENSG mappings: {len(symbol_to_ensg)}')

In [ ]:
seq_banks = pert_embd_dir / 'seq_banks'
target_banks = pert_embd_dir / 'target_banks'

with open(seq_banks / 'dna_to_idx.json') as f:
    dna_to_idx = json.load(f)

with open(seq_banks / 'adamson_gene_to_dna_idx.json') as f:
    adamson_gene_to_dna_idx = json.load(f)

with open(seq_banks / 'norman_guide_to_dna_idx.json') as f:
    norman_guide_to_dna_idx = json.load(f)

with open(target_banks / 'gene_to_target_idx.json') as f:
    gene_to_target_idx = json.load(f)

drug_to_chem_idx = {}
if (seq_banks / 'drug_to_chem_idx.json').exists():
    with open(seq_banks / 'drug_to_chem_idx.json') as f:
        drug_to_chem_idx = json.load(f)

print(f'DNA embeddings: {len(dna_to_idx)} sgID_AB + {len(adamson_gene_to_dna_idx)} adamson + {len(norman_guide_to_dna_idx)} norman')
print(f'Target embeddings: {len(gene_to_target_idx)}')
print(f'Chemical embeddings: {len(drug_to_chem_idx)}')

In [ ]:
datasets = {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e': ref_dir / 'rep1e' / 'rpe1_raw_singlecell_01.h5ad',
    'k562gw': ref_dir / 'k562gw' / 'perturb_processed.h5ad',
    'adamson': ref_dir / 'adamson' / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad',
    'norman': ref_dir / 'norman' / 'NormanWeissman2019_filtered.h5ad',
    'sciplex': ref_dir / 'sciplex' / 'SrivatsanTrapnell2020_sciplex3.h5ad',
}

## Helper Functions

In [ ]:
def is_valid(val):
    if pd.isna(val):
        return False
    if isinstance(val, str) and (val.strip() == '' or val.strip().lower() == 'nan'):
        return False
    return True

def clean_gears_name(name):
    if not is_valid(name):
        return None
    name = str(name)
    if name.endswith('+ctrl'):
        return name.replace('+ctrl', '')
    if name.startswith('ctrl+'):
        return name.replace('ctrl+', '')
    if name.lower() == 'ctrl':
        return 'control'
    return name.strip()

In [ ]:
def normalize_chunk(X, total_counts=None):
    if total_counts is None:
        total_counts = X.sum(axis=1, keepdims=True)
    total_counts = np.maximum(total_counts, 1)
    X_norm = X / total_counts * COUNT_NORMALIZE_TARGET
    return np.log1p(X_norm)

def map_to_universe(expr_row, dataset_mask, dataset_gene_indices):
    out = np.zeros(N_GENES, dtype=np.float32)
    out[dataset_mask] = expr_row[dataset_gene_indices]
    return out

In [ ]:
def get_dataset_gene_mapping(adata, gene_to_idx):
    var_df = adata.var
    
    if var_df.index[0].startswith('ENSG'):
        ensg_col = var_df.index
    elif 'ensembl_id' in var_df.columns:
        ensg_col = var_df['ensembl_id']
    elif 'ensemble_id' in var_df.columns:
        ensg_col = var_df['ensemble_id']
    else:
        ensg_col = var_df.index
    
    ds_gene_to_local_idx = {}
    for i, ensg in enumerate(ensg_col):
        if is_valid(ensg) and ensg in gene_to_idx:
            ds_gene_to_local_idx[ensg] = i
    
    universe_mask = np.zeros(N_GENES, dtype=bool)
    local_indices = []
    
    for ensg, local_idx in ds_gene_to_local_idx.items():
        universe_idx = gene_to_idx[ensg]
        universe_mask[universe_idx] = True
        local_indices.append(local_idx)
    
    local_indices = np.array(local_indices)
    return universe_mask, local_indices, ds_gene_to_local_idx

In [ ]:
def build_control_bank(adata, batch_col, condition_col, universe_mask, local_indices):
    control_bank = {}
    
    if condition_col not in adata.obs.columns:
        print(f'  Warning: {condition_col} not in obs, trying alternatives')
        for alt in ['perturbation', 'gene', 'product_name']:
            if alt in adata.obs.columns:
                condition_col = alt
                break
    
    obs = adata.obs
    ctrl_keywords = ['ctrl', 'control', 'vehicle', 'dmso']
    ctrl_mask = obs[condition_col].astype(str).str.lower().isin(ctrl_keywords)
    
    if batch_col not in obs.columns:
        batch_col = 'gem_group' if 'gem_group' in obs.columns else None
    
    if batch_col is None:
        batches = ['all']
        batch_values = pd.Series(['all'] * len(obs))
    else:
        batches = obs[batch_col].unique()
        batch_values = obs[batch_col]
    
    for batch in batches:
        if batch_col is None:
            batch_ctrl = ctrl_mask
        else:
            batch_ctrl = ctrl_mask & (batch_values == batch)
        
        indices = np.where(batch_ctrl)[0]
        if len(indices) == 0:
            continue
        
        X_ctrl = adata.X[indices]
        if issparse(X_ctrl):
            X_ctrl = X_ctrl.toarray()
        
        totals = X_ctrl.sum(axis=1)
        X_norm = normalize_chunk(X_ctrl, totals.reshape(-1, 1))
        
        X_mapped = np.zeros((len(indices), N_GENES), dtype=np.float32)
        X_mapped[:, universe_mask] = X_norm[:, local_indices]
        
        control_bank[batch] = {
            'X': X_mapped,
            'total': np.log1p(totals).astype(np.float32)
        }
    
    print(f'  Built control bank with {len(control_bank)} batches, {sum(len(v["X"]) for v in control_bank.values())} cells')
    return control_bank, batch_col, condition_col

In [ ]:
def write_training_shard(buffer, shard_idx, ds_name, split, gene_mask):
    path = training_dir / split / f'shard_{ds_name}_{split}_{shard_idx:04d}.npz'
    
    np.savez(path,
        control=np.array(buffer['control'], dtype=np.float32),
        control_total=np.array(buffer['control_total'], dtype=np.float32),
        case=np.array(buffer['case'], dtype=np.float32),
        case_total=np.array(buffer['case_total'], dtype=np.float32),
        seq_idx=np.array(buffer['seq_idx'], dtype=np.int32),
        target_idx=np.array(buffer['target_idx'], dtype=np.int32),
        modality=np.array(buffer['modality'], dtype=np.int8),
        mode=np.array(buffer['mode'], dtype=np.int8),
        has_seq=np.array(buffer['has_seq'], dtype=np.bool_),
        has_target=np.array(buffer['has_target'], dtype=np.bool_),
        n_perts=np.array(buffer['n_perts'], dtype=np.int8),
        dose=np.array(buffer['dose'], dtype=np.float32),
        batch_id=np.array(buffer['batch_id'], dtype=np.int32),
        cell_type=np.array(buffer['cell_type'], dtype=np.int8),
        gene_mask=gene_mask.astype(np.bool_)
    )
    return path

def write_pretraining_shard(X, totals, gene_mask, shard_idx, ds_name, split):
    path = pretraining_dir / split / f'pt_{ds_name}_{split}_{shard_idx:04d}.npz'
    np.savez(path,
        x=X.astype(np.float32),
        total=totals.astype(np.float32),
        gene_mask=gene_mask.astype(np.bool_)
    )
    return path

In [ ]:
def empty_buffer():
    return {
        'control': [], 'control_total': [], 'case': [], 'case_total': [],
        'seq_idx': [], 'target_idx': [], 'modality': [], 'mode': [],
        'has_seq': [], 'has_target': [], 'n_perts': [], 'dose': [],
        'batch_id': [], 'cell_type': []
    }

## Process k562e_raw (Primary Dataset)

In [ ]:
ds_name = 'k562e_raw'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits[ds_name]
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

In [ ]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'gene', universe_mask, local_indices
)

In [ ]:
def get_split_for_pert(pert, train_perts, val_perts, test_perts):
    if pert in train_perts:
        return 'train'
    elif pert in val_perts:
        return 'val'
    elif pert in test_perts:
        return 'test'
    return None

In [ ]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': []}
pt_totals = {'train': [], 'val': []}
pt_shard_counts = {'train': 0, 'val': 0}

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        pert = clean_gears_name(row.get('gene', row.get('condition', '')))
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert is None or pert == 'control'
        is_holdout = pert in holdout_perts if pert else False
        
        if not is_holdout:
            pt_split = 'train' if np.random.random() < 0.9 else 'val'
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        
        if is_control:
            continue
        
        split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get('gem_group', 'all')
        if batch not in control_bank:
            continue
        
        sgid_ab = str(row.get('sgID_AB', '')).replace(',', '-').strip()
        seq_idx = dna_to_idx.get(sgid_ab, -1)
        target_idx = gene_to_target_idx.get(pert, gene_to_target_idx.get(row.get('gene_id', ''), -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]} shards')

In [ ]:
del adata, control_bank
gc.collect()

## Process rep1e Dataset (CRISPRi, Large)

In [ ]:
ds_name = 'rep1e'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')
print(f'  Obs columns: {list(adata.obs.columns)[:15]}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

In [ ]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'gene', universe_mask, local_indices
)

In [ ]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': []}
pt_totals = {'train': [], 'val': []}
pt_shard_counts = {'train': 0, 'val': 0}

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        pert = clean_gears_name(row.get('gene', row.get('condition', '')))
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert is None or pert == 'control'
        is_holdout = pert in holdout_perts if pert else False
        
        if not is_holdout:
            pt_split = 'train' if np.random.random() < 0.9 else 'val'
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        
        if is_control:
            continue
        
        split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get('gem_group', 'all')
        if batch not in control_bank:
            continue
        
        sgid_ab = str(row.get('sgID_AB', '')).replace(',', '-').strip()
        seq_idx = dna_to_idx.get(sgid_ab, -1)
        target_idx = gene_to_target_idx.get(pert, gene_to_target_idx.get(row.get('gene_id', ''), -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('RPE1', 1))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]} shards')

In [ ]:
del adata, control_bank
gc.collect()

## Process k562gw Dataset (CRISPRi, Genome-Wide)

In [ ]:
ds_name = 'k562gw'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')
print(f'  Obs columns: {list(adata.obs.columns)[:15]}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

In [ ]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'gene', universe_mask, local_indices
)

In [ ]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': []}
pt_totals = {'train': [], 'val': []}
pt_shard_counts = {'train': 0, 'val': 0}

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        pert = clean_gears_name(row.get('gene', row.get('condition', '')))
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert is None or pert == 'control'
        is_holdout = pert in holdout_perts if pert else False
        
        if not is_holdout:
            pt_split = 'train' if np.random.random() < 0.9 else 'val'
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        
        if is_control:
            continue
        
        split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get('gem_group', 'all')
        if batch not in control_bank:
            continue
        
        sgid_ab = str(row.get('sgID_AB', '')).replace(',', '-').strip()
        seq_idx = dna_to_idx.get(sgid_ab, -1)
        target_idx = gene_to_target_idx.get(pert, gene_to_target_idx.get(row.get('gene_id', ''), -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]} shards')

In [ ]:
del adata, control_bank
gc.collect()

## Process Adamson Dataset

In [ ]:
ds_name = 'adamson'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

In [ ]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'perturbation', universe_mask, local_indices
)

In [ ]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': []}
pt_totals = {'train': [], 'val': []}
pt_shard_counts = {'train': 0, 'val': 0}

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        pert = str(row.get('perturbation', row.get('condition', ''))).strip()
        pert_clean = pert.split('/')[0].strip() if '/' in pert else pert
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = pert.lower() in ['control', 'ctrl', 'nan', '']
        is_holdout = pert_clean in holdout_perts or pert in holdout_perts
        
        if not is_holdout:
            pt_split = 'train' if np.random.random() < 0.9 else 'val'
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        
        if is_control:
            continue
        
        split = get_split_for_pert(pert_clean, train_perts, val_perts, test_perts)
        if split is None:
            split = get_split_for_pert(pert, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get(batch_col, 'all') if batch_col else 'all'
        if batch not in control_bank:
            batch = list(control_bank.keys())[0] if control_bank else None
        if batch is None:
            continue
        
        seq_idx = adamson_gene_to_dna_idx.get(pert, adamson_gene_to_dna_idx.get(pert_clean, -1))
        target_idx = gene_to_target_idx.get(pert_clean, gene_to_target_idx.get(pert, -1))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([0, -1, -1, -1])
        buf['mode'].append([0, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]} shards')

In [ ]:
del adata, control_bank
gc.collect()

## Process Norman Dataset (CRISPRa, dual-gene)

In [ ]:
ds_name = 'norman'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')
print(f'  Obs columns: {adata.obs.columns.tolist()}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

In [ ]:
norman_target_df = pd.read_csv(ref_dir / 'norman' / 'norman_guideid_ensemble_id_map.csv')

guide_to_ensg = {}
for _, row in norman_target_df.iterrows():
    guide_id = row['guide_id']
    if is_valid(guide_id):
        first_ensg = row.get('first_id')
        second_ensg = row.get('second_id')
        guide_to_ensg[guide_id] = (first_ensg if is_valid(first_ensg) else None,
                                   second_ensg if is_valid(second_ensg) else None)

In [ ]:
control_bank, batch_col, condition_col = build_control_bank(
    adata, 'gem_group', 'guide_id', universe_mask, local_indices
)

In [ ]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': []}
pt_totals = {'train': [], 'val': []}
pt_shard_counts = {'train': 0, 'val': 0}

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        guide_id = str(row.get('guide_id', '')).strip()
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = guide_id.lower() in ['control', 'ctrl', 'nan', ''] or 'negctrl' in guide_id.lower()
        
        parts = guide_id.split('_')
        gene_a = parts[0] if len(parts) >= 1 else None
        gene_b = parts[1] if len(parts) >= 2 and parts[1].lower() != 'negctrl0' else None
        
        is_holdout = (gene_a in holdout_perts if gene_a else False) or (gene_b in holdout_perts if gene_b else False)
        
        if not is_holdout:
            pt_split = 'train' if np.random.random() < 0.9 else 'val'
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        
        if is_control:
            continue
        
        split = get_split_for_pert(guide_id, train_perts, val_perts, test_perts)
        if split is None and gene_a:
            split = get_split_for_pert(gene_a, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get(batch_col, 'all') if batch_col else 'all'
        if batch not in control_bank:
            batch = list(control_bank.keys())[0] if control_bank else None
        if batch is None:
            continue
        
        seq_idx_a = norman_guide_to_dna_idx.get(guide_id, norman_guide_to_dna_idx.get(gene_a, -1))
        
        ensg_a, ensg_b = guide_to_ensg.get(guide_id, (None, None))
        target_idx_a = gene_to_target_idx.get(ensg_a, gene_to_target_idx.get(gene_a, -1)) if ensg_a or gene_a else -1
        target_idx_b = gene_to_target_idx.get(ensg_b, gene_to_target_idx.get(gene_b, -1)) if ensg_b or gene_b else -1
        
        n_perts = 2 if gene_b and target_idx_b >= 0 else 1
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        
        if n_perts == 2:
            buf['seq_idx'].append([seq_idx_a, seq_idx_a, -1, -1])
            buf['target_idx'].append([target_idx_a, target_idx_b, -1, -1])
            buf['modality'].append([0, 0, -1, -1])
            buf['mode'].append([1, 1, -1, -1])
            buf['has_seq'].append([seq_idx_a >= 0, seq_idx_a >= 0, False, False])
            buf['has_target'].append([target_idx_a >= 0, target_idx_b >= 0, False, False])
        else:
            buf['seq_idx'].append([seq_idx_a, -1, -1, -1])
            buf['target_idx'].append([target_idx_a, -1, -1, -1])
            buf['modality'].append([0, -1, -1, -1])
            buf['mode'].append([1, -1, -1, -1])
            buf['has_seq'].append([seq_idx_a >= 0, False, False, False])
            buf['has_target'].append([target_idx_a >= 0, False, False, False])
        
        buf['n_perts'].append(n_perts)
        buf['dose'].append([-1.0, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_to_id.get('K562', 0))
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]} shards')

In [ ]:
del adata, control_bank
gc.collect()

## Process Sciplex Dataset (Chemical Perturbations)

In [ ]:
ds_name = 'sciplex'
print(f'Processing {ds_name}...')

adata = sc.read_h5ad(datasets[ds_name], backed='r')
print(f'  Shape: {adata.shape}')
print(f'  Obs columns: {adata.obs.columns.tolist()[:15]}')

universe_mask, local_indices, ds_gene_map = get_dataset_gene_mapping(adata, gene_to_idx)
print(f'  Genes in universe: {universe_mask.sum()}')

splits = dataset_splits.get(ds_name, {'train': [], 'val': [], 'test': []})
train_perts = set(splits['train'])
val_perts = set(splits['val'])
test_perts = set(splits['test'])

In [ ]:
pert_col = 'product_name' if 'product_name' in adata.obs.columns else 'perturbation'
dose_col = 'dose' if 'dose' in adata.obs.columns else None
cell_type_col = 'cell_type' if 'cell_type' in adata.obs.columns else None

control_bank, batch_col, condition_col = build_control_bank(
    adata, 'plate', pert_col, universe_mask, local_indices
)

In [ ]:
target_col = 'target' if 'target' in adata.obs.columns else None
print(f'  Target column: {target_col}')

drug_to_target = {}
if target_col:
    for drug in adata.obs[pert_col].dropna().unique():
        if str(drug).lower() in ['control', 'vehicle', 'dmso', 'nan', '']:
            continue
        drug_rows = adata.obs[adata.obs[pert_col] == drug]
        targets = drug_rows[target_col].dropna().unique()
        if len(targets) > 0:
            target_name = str(targets[0]).strip()
            if target_name and target_name.lower() not in ['nan', '']:
                drug_to_target[drug] = target_name

print(f'  Drug -> target mappings: {len(drug_to_target)}')

In [ ]:
buffers = {'train': empty_buffer(), 'val': empty_buffer(), 'test': empty_buffer()}
shard_counts = {'train': 0, 'val': 0, 'test': 0}
pt_buffers = {'train': [], 'val': []}
pt_totals = {'train': [], 'val': []}
pt_shard_counts = {'train': 0, 'val': 0}

n_cells = adata.n_obs
batch_id_map = {b: i for i, b in enumerate(control_bank.keys())}

for start in tqdm(range(0, n_cells, CHUNK_SIZE), desc=f'{ds_name}'):
    end = min(start + CHUNK_SIZE, n_cells)
    
    chunk_obs = adata.obs.iloc[start:end].copy()
    chunk_X = adata.X[start:end]
    if issparse(chunk_X):
        chunk_X = chunk_X.toarray()
    
    chunk_totals = chunk_X.sum(axis=1)
    chunk_X_norm = normalize_chunk(chunk_X, chunk_totals.reshape(-1, 1))
    chunk_log_totals = np.log1p(chunk_totals)
    
    for i, (idx, row) in enumerate(chunk_obs.iterrows()):
        drug = str(row.get(pert_col, '')).strip()
        
        expr_mapped = np.zeros(N_GENES, dtype=np.float32)
        expr_mapped[universe_mask] = chunk_X_norm[i, local_indices]
        
        is_control = drug.lower() in ['control', 'vehicle', 'dmso', 'nan', '']
        is_holdout = drug in holdout_perts if drug else False
        
        if not is_holdout:
            pt_split = 'train' if np.random.random() < 0.9 else 'val'
            pt_buffers[pt_split].append(expr_mapped)
            pt_totals[pt_split].append(chunk_log_totals[i])
            
            if len(pt_buffers[pt_split]) >= PT_SHARD_SIZE:
                write_pretraining_shard(
                    np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
                    universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
                )
                pt_buffers[pt_split] = []
                pt_totals[pt_split] = []
                pt_shard_counts[pt_split] += 1
        
        if is_control:
            continue
        
        split = get_split_for_pert(drug, train_perts, val_perts, test_perts)
        if split is None:
            continue
        
        batch = row.get(batch_col, 'all') if batch_col else 'all'
        if batch not in control_bank:
            batch = list(control_bank.keys())[0] if control_bank else None
        if batch is None:
            continue
        
        seq_idx = drug_to_chem_idx.get(drug, -1)
        target_name = drug_to_target.get(drug)
        if target_name:
            target_ensg = symbol_to_ensg.get(target_name)
            target_idx = gene_to_target_idx.get(target_ensg, gene_to_target_idx.get(target_name, -1))
        else:
            target_idx = -1
        
        dose_val = float(row.get(dose_col, -1)) if dose_col else -1.0
        
        cell_type_name = str(row.get(cell_type_col, 'unknown')) if cell_type_col else 'unknown'
        cell_type_id = cell_type_to_id.get(cell_type_name, cell_type_to_id.get('unknown', 4))
        
        ctrl = control_bank[batch]
        ri = np.random.randint(len(ctrl['X']))
        
        buf = buffers[split]
        buf['control'].append(ctrl['X'][ri])
        buf['control_total'].append(ctrl['total'][ri])
        buf['case'].append(expr_mapped)
        buf['case_total'].append(chunk_log_totals[i])
        buf['seq_idx'].append([seq_idx, -1, -1, -1])
        buf['target_idx'].append([target_idx, -1, -1, -1])
        buf['modality'].append([2, -1, -1, -1])
        buf['mode'].append([4, -1, -1, -1])
        buf['has_seq'].append([seq_idx >= 0, False, False, False])
        buf['has_target'].append([target_idx >= 0, False, False, False])
        buf['n_perts'].append(1)
        buf['dose'].append([dose_val, -1.0, -1.0, -1.0])
        buf['batch_id'].append(batch_id_map.get(batch, 0))
        buf['cell_type'].append(cell_type_id)
        
        if len(buf['case']) >= SHARD_SIZE:
            write_training_shard(buf, shard_counts[split], ds_name, split, universe_mask)
            buffers[split] = empty_buffer()
            shard_counts[split] += 1
    
    del chunk_X, chunk_X_norm, chunk_obs
    gc.collect()

for split in ['train', 'val', 'test']:
    if buffers[split]['case']:
        write_training_shard(buffers[split], shard_counts[split], ds_name, split, universe_mask)
        shard_counts[split] += 1

for pt_split in ['train', 'val']:
    if pt_buffers[pt_split]:
        write_pretraining_shard(
            np.array(pt_buffers[pt_split]), np.array(pt_totals[pt_split]),
            universe_mask, pt_shard_counts[pt_split], ds_name, pt_split
        )
        pt_shard_counts[pt_split] += 1

print(f'{ds_name} complete: train={shard_counts["train"]}, val={shard_counts["val"]}, test={shard_counts["test"]} shards')
print(f'  Pretraining: train={pt_shard_counts["train"]}, val={pt_shard_counts["val"]} shards')

In [ ]:
del adata, control_bank
gc.collect()

## Generate Alignment Pairs

In [ ]:
alignment_pairs = {'train': [], 'val': [], 'test': []}

for sgid_ab, seq_idx in dna_to_idx.items():
    parts = sgid_ab.split('|')[0].split('_')
    gene = parts[0] if parts else None
    
    if gene is None:
        continue
    
    target_idx = gene_to_target_idx.get(gene, -1)
    if target_idx < 0:
        continue
    
    k562e_splits = dataset_splits.get('k562e_raw', {})
    if gene in k562e_splits.get('train', []):
        split = 'train'
    elif gene in k562e_splits.get('val', []):
        split = 'val'
    elif gene in k562e_splits.get('test', []):
        split = 'test'
    else:
        split = 'train'
    
    alignment_pairs[split].append({
        'seq_idx': seq_idx,
        'target_idx': target_idx,
        'modality': 0,
        'mode': 0,
        'gene_name': gene
    })

for gene, seq_idx in adamson_gene_to_dna_idx.items():
    gene_clean = gene.split('/')[0].strip() if '/' in gene else gene
    target_idx = gene_to_target_idx.get(gene_clean, gene_to_target_idx.get(gene, -1))
    
    if target_idx < 0:
        continue
    
    adamson_splits = dataset_splits.get('adamson', {})
    if gene in adamson_splits.get('train', []) or gene_clean in adamson_splits.get('train', []):
        split = 'train'
    elif gene in adamson_splits.get('val', []) or gene_clean in adamson_splits.get('val', []):
        split = 'val'
    elif gene in adamson_splits.get('test', []) or gene_clean in adamson_splits.get('test', []):
        split = 'test'
    else:
        split = 'train'
    
    alignment_pairs[split].append({
        'seq_idx': seq_idx,
        'target_idx': target_idx,
        'modality': 0,
        'mode': 0,
        'gene_name': gene_clean
    })

for guide_id, seq_idx in norman_guide_to_dna_idx.items():
    parts = guide_id.split('_')
    gene_a = parts[0] if parts else None
    
    if gene_a is None:
        continue
    
    target_idx = gene_to_target_idx.get(gene_a, -1)
    if target_idx < 0:
        continue
    
    norman_splits = dataset_splits.get('norman', {})
    if guide_id in norman_splits.get('train', []) or gene_a in norman_splits.get('train', []):
        split = 'train'
    elif guide_id in norman_splits.get('val', []) or gene_a in norman_splits.get('val', []):
        split = 'val'
    elif guide_id in norman_splits.get('test', []) or gene_a in norman_splits.get('test', []):
        split = 'test'
    else:
        split = 'train'
    
    alignment_pairs[split].append({
        'seq_idx': seq_idx,
        'target_idx': target_idx,
        'modality': 0,
        'mode': 1,
        'gene_name': gene_a
    })

chem_align_count = 0
chem_target_found = 0
for drug, chem_idx in drug_to_chem_idx.items():
    target_name = drug_to_target.get(drug)
    if target_name is None:
        continue
    
    target_ensg = symbol_to_ensg.get(target_name)
    if target_ensg:
        target_idx = gene_to_target_idx.get(target_ensg, -1)
    else:
        target_idx = gene_to_target_idx.get(target_name, -1)
    
    if target_idx < 0:
        continue
    
    chem_target_found += 1
    
    sciplex_splits = dataset_splits.get('sciplex', {})
    if drug in sciplex_splits.get('train', []):
        split = 'train'
    elif drug in sciplex_splits.get('val', []):
        split = 'val'
    elif drug in sciplex_splits.get('test', []):
        split = 'test'
    else:
        split = 'train'
    
    alignment_pairs[split].append({
        'seq_idx': chem_idx,
        'target_idx': target_idx,
        'modality': 2,
        'mode': 4,
        'gene_name': drug
    })
    chem_align_count += 1

print(f'Alignment pairs: train={len(alignment_pairs["train"])}, val={len(alignment_pairs["val"])}, test={len(alignment_pairs["test"])}')
print(f'  Chemical: {chem_align_count} pairs ({chem_target_found} drugs with valid target_idx)')

In [ ]:
for split in ['train', 'val', 'test']:
    pairs = alignment_pairs[split]
    if not pairs:
        continue
    
    filtered = [p for p in pairs if p['gene_name'] not in holdout_perts] if split != 'test' else pairs
    
    path = pert_embd_dir / split / f'align_{split}.npz'
    np.savez(path,
        seq_idx=np.array([p['seq_idx'] for p in filtered], dtype=np.int32),
        target_idx=np.array([p['target_idx'] for p in filtered], dtype=np.int32),
        modality=np.array([p['modality'] for p in filtered], dtype=np.int8),
        mode=np.array([p['mode'] for p in filtered], dtype=np.int8)
    )
    print(f'Wrote {len(filtered)} alignment pairs to {path}')

## Verify and Extend input_to_id.json

The `input_to_id.json` file maps "GENE_mode_dataset" -> seq_idx for pathway evals. It's primarily generated in Notebook 2, but we verify and extend it here.

In [ ]:
input_to_id_path = pert_embd_dir / 'input_to_id.json'

if input_to_id_path.exists():
    with open(input_to_id_path) as f:
        input_to_id = json.load(f)
    print(f'Loaded input_to_id.json with {len(input_to_id)} entries')
    
    crispri_count = sum(1 for k in input_to_id if '_crispri_' in k)
    crispra_count = sum(1 for k in input_to_id if '_crispra_' in k)
    inhibitor_count = sum(1 for k in input_to_id if '_inhibitor_' in k)
    print(f'  CRISPRi entries: {crispri_count}')
    print(f'  CRISPRa entries: {crispra_count}')
    print(f'  Inhibitor entries: {inhibitor_count}')
else:
    print(f'Warning: {input_to_id_path} not found. Creating from alignment pairs.')
    input_to_id = {}
    
    mode_names = {0: 'crispri', 1: 'crispra', 4: 'inhibitor'}
    ds_by_mode = {0: 'k562e', 1: 'norman', 4: 'sciplex'}
    
    for split, pairs in alignment_pairs.items():
        for p in pairs:
            mode = p['mode']
            gene = p['gene_name']
            seq_idx = p['seq_idx']
            mode_name = mode_names.get(mode, 'unknown')
            ds_name = ds_by_mode.get(mode, 'unknown')
            key = f'{gene}_{mode_name}_{ds_name}'
            if key not in input_to_id:
                input_to_id[key] = seq_idx
    
    with open(input_to_id_path, 'w') as f:
        json.dump(input_to_id, f, indent=2)
    print(f'Created input_to_id.json with {len(input_to_id)} entries')

## Summary and Validation

In [ ]:
print('=== Data Prep Notebook 3 Complete ===')
print('\nTraining shards:')
for split in ['train', 'val', 'test']:
    shards = list((training_dir / split).glob('shard_*.npz'))
    print(f'  {split}: {len(shards)} shards')

print('\nPretraining shards:')
for split in ['train', 'val']:
    shards = list((pretraining_dir / split).glob('pt_*.npz'))
    print(f'  {split}: {len(shards)} shards')

print('\nAlignment pairs:')
for split in ['train', 'val', 'test']:
    align_file = pert_embd_dir / split / f'align_{split}.npz'
    if align_file.exists():
        with np.load(align_file) as data:
            print(f'  {split}: {len(data["seq_idx"])} pairs')

In [ ]:
print('\nValidation - checking first training shard:')
train_shards = list((training_dir / 'train').glob('shard_*.npz'))
if train_shards:
    with np.load(train_shards[0]) as data:
        print(f'  control shape: {data["control"].shape}')
        print(f'  case shape: {data["case"].shape}')
        print(f'  seq_idx shape: {data["seq_idx"].shape}')
        print(f'  target_idx shape: {data["target_idx"].shape}')
        print(f'  n_perts range: {data["n_perts"].min()} - {data["n_perts"].max()}')
        print(f'  gene_mask sum: {data["gene_mask"].sum()}')